# RAG Studio — Capstone Notebook

This notebook exercises the `rag` package end-to-end, live, through the same
`RagStudioPipeline` façade the FastAPI backend and React SPA use:

**ingest → governance → retrieve → rerank → generate → evaluate**

Unlike every property/unit test in this project (which mock all provider I/O),
this notebook makes **real calls** to OpenAI (embeddings, `gpt-4o-mini` generation)
and to the RAGAS / DeepEval judges — the one place live provider calls are meant to
happen, per the project's test-layer convention.

Run this on the `genai2026` kernel with a valid `OPENAI_API_KEY` in `10_RAG/.env`.


In [1]:
import sys, os
from pathlib import Path

# Make the `rag` package importable (this notebook lives in capstone_rag_studio/).
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "rag").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT.parent / ".env")   # 10_RAG/.env — same path backend/main.py loads

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in 10_RAG/.env before running this notebook."
print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))


OPENAI_API_KEY loaded: True


## 1. Ingest

We ingest three small documents under two different Access_Policies:
- Two **PUBLIC** documents about Paris and about Tokyo.
- One **RESTRICTED** document (an internal HR memo) visible only to `hr-team`.

This lets us later demonstrate governance filtering with two different
Acting_Principals.


In [2]:
import tempfile

from rag.pipeline import RagStudioPipeline
from rag.interfaces import AccessPolicy, Principal

workdir = Path(tempfile.mkdtemp(prefix="rag_studio_capstone_"))
pipeline = RagStudioPipeline(db_path=str(workdir / "db.sqlite"), upload_dir=str(workdir / "uploads"))

docs_to_write = {
    "paris.txt": "Paris is the capital of France. The Eiffel Tower is a famous landmark in Paris, "
                 "built in 1889 for the World's Fair.",
    "tokyo.txt": "Tokyo is the capital of Japan. It is one of the most populous metropolitan areas "
                 "in the world and is known for its blend of traditional and modern culture.",
    "hr_memo.txt": "CONFIDENTIAL HR MEMO: Employee salaries will be reviewed in Q3. "
                   "This document is restricted to the HR team only.",
}
for name, text in docs_to_write.items():
    (workdir / name).write_text(text)

# Ingest the two public documents as "carol" (a general content owner).
public_docs = pipeline.ingest_documents(
    [
        {"path": str(workdir / "paris.txt"), "source": "paris.txt"},
        {"path": str(workdir / "tokyo.txt"), "source": "tokyo.txt"},
    ],
    AccessPolicy(access_level="PUBLIC"),
)

# Ingest the HR memo as RESTRICTED: owned by "carol", also visible to the "hr-team" role.
restricted_docs = pipeline.ingest_documents(
    [{"path": str(workdir / "hr_memo.txt"), "source": "hr_memo.txt"}],
    AccessPolicy(access_level="RESTRICTED", owner="carol", acl=["hr-team"]),
)

print(f"Ingested {len(public_docs)} public doc(s) and {len(restricted_docs)} restricted doc(s).")
assert len(pipeline.all_docs()) == 3


Ingested 2 public doc(s) and 1 restricted doc(s).


## 2. Build two strategy variants

- **baseline**: dense retrieval, no reranking, no query transform.
- **hybrid_rerank**: hybrid RRF retrieval (dense + BM25) with a local cross-encoder
  reranker.

Both use the free local `minilm` embedder so only the generation stage makes a paid
call.


In [3]:
from rag.config import PipelineConfig

baseline_config = PipelineConfig(
    name="baseline",
    embedding="minilm",
    vector_store="memory",
    retrieval="dense",
    reranker="none",
    query_transform="none",
    guardrails="light",
    caching="off",
    llm="gpt-4o-mini",
)

hybrid_config = PipelineConfig(
    name="hybrid_rerank",
    embedding="minilm",
    vector_store="memory",
    retrieval="hybrid_rrf",
    reranker="cross_encoder",
    rerank_top_k=3,
    query_transform="none",
    guardrails="light",
    caching="off",
    llm="gpt-4o-mini",
)

pipeline.save_variant(baseline_config)
pipeline.save_variant(hybrid_config)

for config in (baseline_config, hybrid_config):
    result = pipeline.build_index(config, pipeline.all_docs())
    print(f"{config.name}: indexed {len(result.chunks)} chunks")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


baseline: indexed 3 chunks
hybrid_rerank: indexed 3 chunks


## 3. Run >= 2 variants against a question — cited answers side by side


In [4]:
alice = Principal(id="alice")   # a general user, not on the HR team
question = "What famous landmark is in Paris and when was it built?"

outcome = pipeline.run_all_variants([baseline_config, hybrid_config], question, alice)

assert not outcome.errors, f"Unexpected variant failures: {outcome.errors}"

for name, result in outcome.results.items():
    print(f"--- {name} ---")
    print("Answer:", result.answer)
    print("Citations:", [(c.chunk_id, c.source) for c in result.citations])
    print("Per-stage timing (ms):", {k: round(v, 1) for k, v in result.per_stage_timing.items()})
    print("Governance filtered count:", result.governance_filtered_count)
    print()


Split strings:   0%|          | 0/3 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/3 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/3 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/3 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- baseline ---
Answer: The famous landmark in Paris is the Eiffel Tower, and it was built in 1889 for the World's Fair [paris.txt::0].
Citations: [('paris.txt::0', 'paris.txt')]
Per-stage timing (ms): {'input_guardrail': 0.0, 'query_transform': 0.0, 'retrieval': 492.2, 'governance': 0.0, 'post_retrieval': 0.0, 'rerank': 0.0, 'generate': 2034.3, 'output_guardrail': 0.0}
Governance filtered count: 1

--- hybrid_rerank ---
Answer: The famous landmark in Paris is the Eiffel Tower, and it was built in 1889 for the World's Fair [paris.txt::0].
Citations: [('paris.txt::0', 'paris.txt')]
Per-stage timing (ms): {'input_guardrail': 0.0, 'query_transform': 0.0, 'retrieval': 93.3, 'governance': 0.0, 'post_retrieval': 0.0, 'rerank': 5939.7, 'generate': 1703.3, 'output_guardrail': 0.3}
Governance filtered count: 1



## 4. Governance: the same question under two Acting_Principals

`alice` is not on the HR team, so the restricted HR memo is filtered out of her
authorized context before it ever reaches reranking or generation — the
`Governance_Filter` runs on every retrieval, for every principal, with no exceptions.
`carol` (the doc owner) is authorized to see it. We ask a question that only the
restricted memo can answer, using the `baseline` variant.


In [5]:
carol = Principal(id="carol")   # ingested (and therefore owns) the HR memo above -- see Access_Policy semantics

hr_question = "When will employee salaries be reviewed?"

alice_result = pipeline.run_variant(baseline_config, hr_question, alice)
carol_result = pipeline.run_variant(baseline_config, hr_question, carol)

print("alice (unauthorized) ->", alice_result.answer)
print("  no_accessible_context:", alice_result.no_accessible_context)
print("  governance_filtered_count:", alice_result.governance_filtered_count)
print()
print("carol (owner) ->", carol_result.answer)
print("  no_accessible_context:", carol_result.no_accessible_context)
print("  governance_filtered_count:", carol_result.governance_filtered_count)

# The unauthorized principal must never see the restricted content: governance
# filtered out the HR memo chunk for her (the other, irrelevant public chunks are
# still retrieved since top_k covers the whole small corpus, so the LLM correctly
# says the answer isn't in its -- authorized-only -- context).
assert alice_result.governance_filtered_count >= 1
assert "q3" not in alice_result.answer.lower() and "salaries" not in alice_result.answer.lower()
assert not any(c.source == "hr_memo.txt" for c in alice_result.retrieved_chunks)

# The owner sees the memo, with nothing filtered out.
assert carol_result.governance_filtered_count == 0
assert any(c.source == "hr_memo.txt" for c in carol_result.retrieved_chunks)


alice (unauthorized) -> answer not found in provided context
  no_accessible_context: False
  governance_filtered_count: 1

carol (owner) -> Employee salaries will be reviewed in Q3 [hr_memo.txt::0].
  no_accessible_context: False
  governance_filtered_count: 0


## 5. Evaluation: RAGAS + DeepEval over a Golden_Set

A small Golden_Set with ground-truth references, evaluated by both frameworks.
Reference-free metrics (`faithfulness`, `answer_relevancy`) run over every record;
reference-dependent metrics (`context_precision`, `context_recall`) only run where a
`reference` is present — records without one are excluded and the exclusion is
reported (Req 8.8).


In [6]:
golden_set = [
    {
        "id": "paris-landmark",
        "question": question,
        "answer": outcome.results["hybrid_rerank"].answer,
        "contexts": [c.text for c in outcome.results["hybrid_rerank"].reranked_chunks],
        "reference": "The Eiffel Tower, built in 1889 for the World's Fair.",
    },
    {
        "id": "tokyo-capital",
        "question": "What is the capital of Japan?",
        "answer": "Tokyo is the capital of Japan.",
        "contexts": ["Tokyo is the capital of Japan. It is one of the most populous metropolitan areas in the world."],
        # No reference on purpose -- demonstrates P24 exclusion reporting below.
    },
]

ragas_scores = pipeline.evaluate_golden(
    golden_set, framework_name="ragas",
    metrics=["faithfulness", "answer_relevancy", "context_precision", "context_recall"],
)
print("RAGAS scores:", ragas_scores.scores)
print("RAGAS excluded (no reference):", ragas_scores.excluded_reference_dependent)
assert ragas_scores.excluded_reference_dependent == ["tokyo-capital"]


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

RAGAS scores: {'faithfulness': 1.0, 'answer_relevancy': 0.9926085585470736, 'context_precision': 0.9999999999, 'context_recall': 1.0}
RAGAS excluded (no reference): ['tokyo-capital']


In [7]:
deepeval_scores = pipeline.evaluate_golden(
    golden_set, framework_name="deepeval",
    metrics=["faithfulness", "answer_relevancy"],
)
print("DeepEval scores:", deepeval_scores.scores)
print("DeepEval passed:", deepeval_scores.passed)
print("DeepEval reasons:")
for metric, reason in deepeval_scores.reasons.items():
    print(f"  {metric}: {reason}")


Output()

Output()

Output()

Output()

DeepEval scores: {'faithfulness': 1.0, 'answer_relevancy': 1.0}
DeepEval passed: {'faithfulness': True, 'answer_relevancy': True}
DeepEval reasons:
  faithfulness: all samples passed
  answer_relevancy: all samples passed


## 6. Wrap-up

This notebook ran the full RAG Studio pipeline live: ingestion under mixed
Access_Policies, two strategy variants compared side by side with citations and
per-stage timing, a governance demonstration proving unauthorized content is
excluded for a less-privileged principal, and RAGAS + DeepEval evaluation over a
Golden_Set (including the reference-exclusion behavior). Every assertion above
passed, so this cell executing is the "0 errors" signal for the notebook.


In [8]:
print("Capstone notebook completed with 0 errors.")


Capstone notebook completed with 0 errors.
